# 24. 필터 + 벡터 검색 — PostFilter · PreFilter · AllowList(iterative scan)

4-1 슬라이드 「PreFilter / PostFilter / AllowList」 · 「필터링 선택 가이드」를 pgvector 로 확인한다.

| 전략 | pgvector 에서 | 언제 |
|---|---|---|
| PostFilter | HNSW 로 ef_search 개 뽑은 뒤 WHERE 로 거름 (기본 동작) | 필터가 약할 때 |
| PreFilter | 필터를 먼저 실행하고 남은 행만 전수 비교 (직접 작성) | 필터가 아주 좁을 때 |
| AllowList 비슷 | `hnsw.iterative_scan` : 조건 맞는 행이 모자라면 그래프를 더 탐색 | 그 중간 |
| 부분 인덱스 | `CREATE INDEX … WHERE tag = 7` | 자주 쓰는 고정 필터 |

In [ ]:
from labutil import *

conn = connect()
for kind in ["ivf", "half", "bin", "hnsw_tag7"]:
    conn.execute(f"DROP INDEX IF EXISTS vec_clu_{kind}")
conn.execute("DROP INDEX IF EXISTS vec_clu_tag")
conn.execute("CREATE INDEX IF NOT EXISTS vec_clu_hnsw ON vec_clu USING hnsw (emb vector_l2_ops)")

# 필터 조건별 정답지는 numpy 로 계산한다 (테이블 전체를 한 번 가져온다)
T = q(conn, "SELECT id, cluster, tag, emb FROM vec_clu ORDER BY id")
X = np.stack(T["emb"].to_numpy()); ids = T["id"].to_numpy(); tag = T["tag"].to_numpy(); clu = T["cluster"].to_numpy()
QS = q(conn, "SELECT qid, emb FROM vec_query WHERE ds='clu' ORDER BY qid LIMIT 100")
Q = np.stack(QS["emb"].to_numpy())


def filtered_truth(mask, k=10):
    idx = np.where(mask)[0]
    return [ids[idx[r]] for r in brute_knn(X[idx], Q, min(k, len(idx)))]


def run(where, truth, params=None, k=10):
    """모든 질의에 대해 WHERE 필터 + 벡터 정렬. 반환: recall, 받은 행 수, 지연 ms"""
    found, times = [], []
    with conn.cursor() as cur:
        for i, qv in enumerate(Q):
            p = {"q": qv, **(params(i) if params else {})}
            t = time.perf_counter()
            cur.execute(f"SELECT id FROM vec_clu WHERE {where} ORDER BY emb <-> %(q)s LIMIT {k}", p)
            found.append([r[0] for r in cur.fetchall()])
            times.append(time.perf_counter() - t)
    tr = truth if isinstance(truth, list) else truth
    return {"recall": float(np.mean([len(set(f) & set(t)) / len(t) for f, t in zip(found, tr)])),
            "받은 행": float(np.mean([len(f) for f in found])), "ms": 1000 * float(np.median(times))}

## 1. PostFilter 의 함정 — 결과가 모자란다
`tag` 는 0~99 가 무작위인 속성. `tag < s` 로 필터의 좁기(선택도)를 바꾼다. 아직 tag 에 B-tree 는 없다.

In [ ]:
SEL = [1, 5, 20, 50, 100]                     # tag < s → 전체의 s%
truth = {s: filtered_truth(tag < s) for s in SEL}

rows = []
for s in SEL:
    rows.append({"선택도 %": s, "전략": "HNSW + 사후 필터", **run(f"tag < {s}", truth[s])})
post = pd.DataFrame(rows)

print(explain(conn, "SELECT id FROM vec_clu WHERE tag < 1 ORDER BY emb <-> %s LIMIT 10", (Q[0],)))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].bar(post["선택도 %"].astype(str), post["받은 행"], color="#FF7043")
axes[0].axhline(10, ls="--", color="gray"); axes[0].text(-0.4, 10.3, "요청 LIMIT 10", color="gray")
axes[0].set_xlabel("필터 통과 비율 (%)"); axes[0].set_ylabel("평균 받은 행 수"); axes[0].set_title("ef_search=40 개 중 필터를 통과한 것만 남는다")
axes[1].bar(post["선택도 %"].astype(str), post["recall"], color="#3D484A")
axes[1].set_ylim(0, 1.05); axes[1].set_xlabel("필터 통과 비율 (%)"); axes[1].set_title("recall@10 (필터 조건 안에서의 정답 대비)")
plt.tight_layout(); plt.show()

**관찰:** 필터가 1% 면 HNSW 가 찾은 후보 40개 중 평균 0.4개만 남는다. **LIMIT 10 인데 0~1행.**
오류가 아니라 조용히 적게 온다 — 4-1 슬라이드 "PostFilter: 최종 결과가 아예 없음"이 이것이다.

## 2. PreFilter — 플래너에 맡기면?
B-tree 를 만들어 주면 1% 필터에서는 플래너가 "B-tree 로 거르고 정확히 정렬"을 고를까?

In [ ]:
conn.execute("CREATE INDEX vec_clu_tag ON vec_clu (tag)")
conn.execute("ANALYZE vec_clu")
print(explain(conn, "SELECT id FROM vec_clu WHERE tag < 1 ORDER BY emb <-> %s LIMIT 10", (Q[0],)))

**관찰:** 고르지 않는다. `LIMIT 10` 이 붙으면 플래너는 HNSW 인덱스 스캔이 가장 싸다고 보고, 필터는 나중에 건다.
PreFilter 가 필요하면 **직접 쓴다**: 필터를 먼저 실행하도록 `MATERIALIZED` CTE 로 가둔다.

In [ ]:
PRE = "WITH c AS MATERIALIZED (SELECT id, emb FROM vec_clu WHERE {w}) SELECT id FROM c ORDER BY emb <-> %(q)s LIMIT 10"
print(explain(conn, PRE.format(w="tag < 1").replace("%(q)s", "%s"), (Q[0],)))


def run_sql(sql, truth, k=10):
    found, times = [], []
    with conn.cursor() as cur:
        for qv in Q:
            t = time.perf_counter()
            cur.execute(sql, {"q": qv})
            found.append([r[0] for r in cur.fetchall()])
            times.append(time.perf_counter() - t)
    return {"recall": float(np.mean([len(set(f) & set(t)) / len(t) for f, t in zip(found, truth)])),
            "받은 행": float(np.mean([len(f) for f in found])), "ms": 1000 * float(np.median(times))}


pre = pd.DataFrame([{"선택도 %": s, "전략": "PreFilter (B-tree → 정확 정렬)", **run_sql(PRE.format(w=f"tag < {s}"), truth[s])}
                    for s in SEL])

## 3. iterative scan — 모자라면 그래프를 더 탐색 (pgvector 0.8)

In [ ]:
conn.execute("DROP INDEX vec_clu_tag")          # HNSW 만으로 비교
rows = []
for mode in ["relaxed_order", "strict_order"]:
    conn.execute(f"SET hnsw.iterative_scan = {mode}")
    for s in SEL:
        rows.append({"선택도 %": s, "전략": f"iterative ({mode})", **run(f"tag < {s}", truth[s])})
conn.execute("RESET hnsw.iterative_scan")
it = pd.DataFrame(rows)
allres = pd.concat([post, pre, it])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2), layout="constrained")
colors = {"HNSW + 사후 필터": "#FF7043", "PreFilter (B-tree → 정확 정렬)": "#3D484A",
          "iterative (relaxed_order)": "#009CA6", "iterative (strict_order)": "#84A59D"}
for name, g in allres.groupby("전략", sort=False):
    for ax, col in zip(axes, ["받은 행", "recall", "ms"]):
        ax.plot(g["선택도 %"], g[col], "-o", color=colors[name], label=name)
for ax, col in zip(axes, ["평균 받은 행 (LIMIT 10)", "recall@10", "지연시간 ms"]):
    ax.set_xscale("log"); ax.set_xticks(SEL, [f"{s}%" for s in SEL]); ax.set_xlabel("필터 통과 비율"); ax.set_title(col)
axes[2].set_yscale("log"); axes[0].legend(fontsize=8)
plt.show()
allres.pivot(index="선택도 %", columns="전략", values="recall").round(3)

**관찰:**
- iterative scan 을 켜면 1% 필터에서도 10행을 채운다. 그래프 탐색을 이어 가며 조건 맞는 행을 모은다 (AllowList 와 비슷한 효과)
- 대신 필터가 좁을수록 더 많이 탐색하므로 느려진다. 상한은 `hnsw.max_scan_tuples`(기본 2만)
- PreFilter 는 언제나 정확(recall 1.0)하지만 필터를 통과한 행 수만큼 거리 계산을 한다.
  1% 면 가장 빠르고, 50% 이상이면 전수 비교에 가까워져 가장 느리다
- 좁은 필터는 PreFilter, 중간은 iterative scan, 약한 필터는 사후 필터 → 4-1 「필터링 선택 가이드」 그대로다

## 4. 필터가 벡터와 "같은 편"인가 — 필터 적합성
`cluster` 는 벡터가 실제로 모여 있는 군집 번호다 (선택도 0.5%).
- 가까운 군집으로 필터: 질의 근처에 조건 맞는 행이 몰려 있다
- 먼 군집으로 필터: 조건 맞는 행이 전부 멀리 있다

In [ ]:
near_c = clu[np.searchsorted(ids, [truth[100][i][0] for i in range(len(Q))])]      # 질의의 최근접이 속한 군집
far_c = (near_c + 100) % 200                                                         # 다른 군집
tr_near = [ids[np.where(clu == c)[0][brute_knn(X[clu == c], Q[i], 10)[0]]] for i, c in enumerate(near_c)]
tr_far = [ids[np.where(clu == c)[0][brute_knn(X[clu == c], Q[i], 10)[0]]] for i, c in enumerate(far_c)]

rows = []
for mode in ["off", "relaxed_order"]:
    conn.execute(f"SET hnsw.iterative_scan = {mode}")
    rows.append({"iterative": mode, "필터": "가까운 군집", **run("cluster = %(c)s", tr_near, lambda i: {"c": int(near_c[i])})})
    rows.append({"iterative": mode, "필터": "먼 군집", **run("cluster = %(c)s", tr_far, lambda i: {"c": int(far_c[i])})})
conn.execute("RESET hnsw.iterative_scan")
fit = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for ax, col in zip(axes, ["받은 행", "recall", "ms"]):
    for j, (mode, g) in enumerate(fit.groupby("iterative")):
        ax.bar(np.arange(2) + (j - 0.5) * 0.4, g[col], 0.4, label=f"iterative={mode}",
               color=["#9aa5a8", "#009CA6"][j])
        for x_, v in zip(np.arange(2) + (j - 0.5) * 0.4, g[col]):
            ax.text(x_, v, f"{v:.2g}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(2), ["가까운 군집 필터", "먼 군집 필터"]); ax.set_title(col)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**관찰:** 선택도가 같아도(0.5%) **필터가 질의와 같은 방향이면** 사후 필터로도 잘 채워지고,
반대 방향이면 iterative scan 도 상한(2만 행)까지 헤매다 **못 채우거나 느려진다**.
3부 31번의 "필터 적합성"(필터 집합과 유사도 집합이 어울리는가)을 인덱스 쪽에서 본 모습이다.

## 5. 부분 인덱스 — 자주 쓰는 필터는 그 행만으로 그래프를 따로 만든다
4-1 슬라이드 "사전 필터를 해도 인덱스는 하나라 부분집합 그래프는 불가능" → PG 는 **부분 인덱스**로 가능하다.

In [ ]:
with timer("tag = 7 인 행(약 1%)만으로 HNSW"):
    conn.execute("CREATE INDEX vec_clu_hnsw_tag7 ON vec_clu USING hnsw (emb vector_l2_ops) WHERE tag = 7")
t7 = filtered_truth(tag == 7)
print(explain(conn, "SELECT id FROM vec_clu WHERE tag = 7 ORDER BY emb <-> %s LIMIT 10", (Q[0],)))
part = run("tag = 7", t7)
conn.execute("DROP INDEX vec_clu_hnsw_tag7")
nopart = run("tag = 7", t7)
pd.DataFrame({"전체 HNSW + 사후 필터": nopart, "부분 HNSW (WHERE tag = 7)": part}).T.round(3)

**관찰:** 부분 인덱스는 tag=7 인 1천 행만으로 그래프를 만들었으므로, 사후 필터 없이 바로 10행을 정확히 돌려준다.
대신 필터 값마다 인덱스가 하나씩 필요하다 → 테넌트·카테고리처럼 **값이 적고 고정된** 필터에 쓴다 (값이 많으면 파티셔닝).